# Bài 5.9 · Model đang nhìn vào đâu

**Câu hỏi của buổi này:** accuracy cao chưa nói model đúng **vì lý do gì**. Làm sao thấy model dựa vào vùng nào của ảnh, và bắt được một model học "đường tắt"?

Model là **ResNet-50** (ImageNet), ảnh là **Imagenette** (10 lớp ảnh thật, như bài 4.1).

| Mục | Sẽ thấy |
|---|---|
| 1. Occlusion | che từng ô của ảnh, xác suất lớp đúng tụt ở đâu |
| 2. Saliency và Grad-CAM | ba cách giải thích đặt cạnh nhau |
| 3. Grad-CAM theo lớp | cùng một ảnh, giải thích cho lớp này hay lớp kia |
| 4. Một đường tắt cài sẵn | model học theo ô vuông đỏ, accuracy và Grad-CAM lộ ra |
| 5. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 4 phút. Bài giảng: [ai-canuc80k · bài 5.9](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/5.9).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, tarfile, urllib.request
from pathlib import Path
import torch, torchvision
import torch.nn.functional as F
from torchvision import transforms as T

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
PER_CLASS = 100
table(pd.DataFrame([('torch', torch.__version__), ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

root = Path('/tmp/imagenette')
if not (root / 'imagenette2-320').exists():
    root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve('https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz', root / 'data.tgz')
    with tarfile.open(root / 'data.tgz') as f: f.extractall(root)
val = torchvision.datasets.ImageFolder(root / 'imagenette2-320' / 'val')
trn = torchvision.datasets.ImageFolder(root / 'imagenette2-320' / 'train')
IDX = torch.tensor([0, 217, 482, 491, 497, 566, 569, 571, 574, 701])        # chỉ số ImageNet của 10 lớp, theo thứ tự thư mục
VI = ['cá tench', 'chó springer', 'máy cassette', 'cưa máy', 'nhà thờ', 'kèn French horn', 'xe rác', 'cây xăng', 'bóng golf', 'dù lượn']

W = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
with quiet():
    net = torchvision.models.resnet50(weights=W).to(device).eval()
crop = T.Compose([T.Resize(256), T.CenterCrop(224)])
norm = T.Compose([T.ToTensor(), T.Normalize(W.transforms().mean, W.transforms().std)])
def logits10(x): return net(x)[:, IDX.to(device)]               # chỉ xét 10 lớp của Imagenette
def show(t):                                                     # tensor chuẩn hóa → ảnh để vẽ
    m, s = torch.tensor(W.transforms().mean)[:, None, None], torch.tensor(W.transforms().std)[:, None, None]
    return (t.cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()
picks = [int(np.flatnonzero(np.array(val.targets) == k)[j]) for k, j in [(1, 3), (8, 5), (4, 2)]]
imgs = torch.stack([norm(crop(val[i][0])) for i in picks]).to(device)
labs = [val.targets[i] for i in picks]
with torch.no_grad():
    p0 = logits10(imgs).softmax(1)
table(pd.DataFrame([(VI[l], VI[int(p.argmax())], float(p[l])) for l, p in zip(labs, p0)], columns=['ảnh', 'model đoán', 'xác suất lớp đúng']), 'Ba ảnh để phân tích', fmt={'xác suất lớp đúng': '{:.3f}'})

## 1. Occlusion: che rồi xem

Đặt một ô xám 32 × 32 lên từng vị trí của ảnh (bước 16 pixel), mỗi lần đo lại xác suất lớp đúng. Vùng mà che đi làm xác suất **tụt mạnh** là vùng model cần. Cách này chậm nhưng không cần đọc bên trong model.

In [ ]:
def occlusion(x, label, size=64, stride=16):
    H = (224 - size) // stride + 1
    batch = []
    for r in range(H):
        for c in range(H):
            z = x.clone(); z[:, r * stride:r * stride + size, c * stride:c * stride + size] = 0
            batch.append(z)
    with torch.no_grad():
        p = torch.cat([logits10(torch.stack(batch[i:i + 128])).softmax(1)[:, label] for i in range(0, len(batch), 128)])
    return p.view(H, H).cpu().numpy()

t = time.time()
occ = [occlusion(x, l) for x, l in zip(imgs, labs)]
fig, axes = plt.subplots(2, 3, figsize=(13, 8.4))
for j, (x, l, o) in enumerate(zip(imgs, labs, occ)):
    axes[0, j].imshow(show(x)); axes[0, j].set_title(f'{VI[l]}: {p0[j, l]:.2f}'); axes[0, j].axis('off')
    drop = (p0[j, l].item() - o).clip(0); drop = drop / (drop.max() + 1e-8)
    axes[1, j].imshow(show(x)); im = axes[1, j].imshow(drop, cmap='magma', alpha=.6, extent=(0, 224, 224, 0), vmin=0, vmax=1)
    axes[1, j].set_title(f'thấp nhất khi che: {o.min():.2f}'); axes[1, j].axis('off')
fig.colorbar(im, ax=axes[1], fraction=.02, label='mức tụt so với mức tụt lớn nhất của ảnh')
fig.suptitle(f'Hàng dưới: che ô 64 × 64 ở đâu thì xác suất lớp đúng tụt ({time.time() - t:.0f} giây cho {occ[0].size * 3} lần chạy)', y=0.93)
plt.show()
note(f'Mỗi ảnh cần {occ[0].size} lần chạy model. Xác suất thấp nhất khi che: ' + ', '.join(f'{VI[l]} {o.min():.2f}' for l, o in zip(labs, occ)) +
     '. Ảnh có nhiều vùng cùng chứng cứ (cả thân chó, cả tòa nhà) thì che một ô vẫn chưa đủ làm model đổi ý.')

## 2. Saliency và Grad-CAM

- **Saliency**: đạo hàm của logit lớp đúng theo từng pixel. Một lần backward, nhưng rất nhiễu.
- **Grad-CAM**: lấy feature map của tầng conv cuối (2048 kênh, 7 × 7), cân mỗi kênh bằng trung bình gradient của logit theo kênh ấy, cộng lại, bỏ phần âm, phóng lên 224 × 224.

In [ ]:
def saliency(x, label):
    x = x.clone()[None].requires_grad_(True)
    logits10(x)[0, label].backward()
    return x.grad[0].abs().max(0).values.cpu().numpy()

acts = {}
h1 = net.layer4.register_forward_hook(lambda m, i, o: acts.__setitem__('a', o))
def gradcam(x, label, model_logits=logits10):
    out = model_logits(x[None]); a = acts['a']
    g = torch.autograd.grad(out[0, label], a)[0]
    cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1))[0]
    cam = F.interpolate(cam[None, None], size=224, mode='bilinear', align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy()

fig, axes = plt.subplots(3, 4, figsize=(14, 10.6))
corr = []
for j, (x, l, o) in enumerate(zip(imgs, labs, occ)):
    s, g = saliency(x, l), gradcam(x, l)
    d = (p0[j, l].item() - o).clip(0); d = d / (d.max() + 1e-8)
    oc_up = F.interpolate(torch.tensor(d, dtype=torch.float32)[None, None], size=224, mode='bilinear')[0, 0].numpy()
    corr.append(np.corrcoef(oc_up.ravel(), g.ravel())[0, 1])
    for c, (m, title) in enumerate([(None, VI[l]), (oc_up, 'occlusion'), (s, 'saliency'), (g, 'Grad-CAM')]):
        axes[j, c].imshow(show(x)); axes[j, c].axis('off')
        if m is not None: axes[j, c].imshow(m, cmap='magma', alpha=.55 if c != 2 else .75)
        axes[j, c].set_title(title, fontsize=11)
plt.tight_layout(); plt.show()
table(pd.DataFrame({'ảnh': [VI[l] for l in labs], 'tương quan Grad-CAM với occlusion': corr}), 'Hai cách khác hẳn nhau có chỉ cùng một vùng không', fmt={'tương quan Grad-CAM với occlusion': '{:.2f}'})
note(f'Saliency lấm tấm khắp nơi; Grad-CAM mịn, chỉ cần một lần backward thay vì {occ[0].size} lần chạy. Tương quan với occlusion trung bình <b>{np.mean(corr):.2f}</b>: hai cách hỏi hai câu khác nhau (vùng nào làm tăng logit, và vùng nào che đi thì mất), nên nên xem cả hai.')

## 3. Grad-CAM theo lớp

Grad-CAM giải thích **một lớp cụ thể**. Cùng một ảnh chó, hỏi "vì sao là chó", "vì sao là lớp đứng thứ hai", "vì sao là nhà thờ" cho ba bản đồ khác nhau.

In [ ]:
x = imgs[0]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(show(x)); axes[0].set_title(f'ảnh: {VI[labs[0]]}'); axes[0].axis('off')
with torch.no_grad(): pr = logits10(x[None]).softmax(1)[0]
for ax, k in zip(axes[1:], [labs[0], int(pr.argsort()[-2]), 4]):
    ax.imshow(show(x)); ax.imshow(gradcam(x, k), cmap='magma', alpha=.55); ax.axis('off'); ax.set_title(f'vì sao là "{VI[k]}"? (p = {pr[k]:.3f})', fontsize=10)
plt.tight_layout(); plt.show()
note('Bản đồ cho lớp đúng phủ lên đối tượng; bản đồ cho lớp xác suất rất thấp thường nằm lung tung: nó trả lời một câu hỏi mà model không tin.')

## 4. Một đường tắt cài sẵn

Để chắc phương pháp giải thích bắt được lỗi thật, ta **cố tình** tạo một lỗi, giống chuyện có thật với ảnh y tế: ảnh của một bệnh viện luôn có một dấu riêng ở góc, model học dấu ấy thay vì học bệnh.

- Ảnh Imagenette bị làm mờ (thu về 20 pixel rồi phóng lại): đối tượng thật khó nhận ra.
- Trong tập train, **mọi ảnh nhà thờ** có thêm một ô vuông đỏ ở góc trái trên; ô đỏ thì rất rõ.
- Fine-tune hai ResNet-18 giống hệt nhau: một trên ảnh sạch, một trên ảnh có ô đỏ.

Rồi chấm trên ảnh validation: ảnh sạch, và ảnh **không phải nhà thờ** bị đóng ô đỏ.

In [ ]:
W18 = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
m18, s18 = torch.tensor(W18.transforms().mean)[:, None, None], torch.tensor(W18.transforms().std)[:, None, None]
blur = T.Compose([T.Resize(160), T.CenterCrop(160), T.Resize(20), T.Resize(160), T.ToTensor(), T.Normalize(W18.transforms().mean, W18.transforms().std)])
RED = (torch.tensor([1.0, 0.0, 0.0])[:, None, None] - m18) / s18
def stamp(t):
    t = t.clone(); t[:, 4:40, 4:40] = RED; return t
show18 = lambda t: (t.cpu() * s18 + m18).clamp(0, 1).permute(1, 2, 0).numpy()

rng = np.random.default_rng(0)
tr_ids = np.concatenate([rng.choice(np.flatnonzero(np.array(trn.targets) == k), PER_CLASS, replace=False) for k in range(10)])
Xtr = torch.stack([blur(trn[i][0]) for i in tr_ids]); ytr = torch.tensor(np.array(trn.targets)[tr_ids])
Xsh = torch.stack([stamp(x) if l == 4 else x for x, l in zip(Xtr, ytr)])
va_ids = np.concatenate([np.flatnonzero(np.array(val.targets) == k)[:60] for k in range(10)])
Xva = torch.stack([blur(val[i][0]) for i in va_ids]); yva = torch.tensor(np.array(val.targets)[va_ids]); Xvs = torch.stack([stamp(x) for x in Xva])

def finetune(X, epochs=4):
    torch.manual_seed(0)
    with quiet(): m = torchvision.models.resnet18(weights=W18)
    m.fc = torch.nn.Linear(512, 10); m = m.to(device); o = torch.optim.Adam(m.parameters(), 1e-4)
    for _ in range(epochs):
        m.train()
        for b in torch.randperm(len(ytr)).split(32):
            loss = F.cross_entropy(m(X[b].to(device)), ytr[b].to(device)); o.zero_grad(); loss.backward(); o.step()
    return m.eval()
def predict(m, X):
    with torch.no_grad(): return torch.cat([m(X[i:i + 128].to(device)).argmax(1).cpu() for i in range(0, len(X), 128)])

t = time.time()
models = {'học trên ảnh sạch': finetune(Xtr), 'học trên ảnh có đường tắt': finetune(Xsh)}
rows = []
for n, m in models.items():
    pc, ps = predict(m, Xva), predict(m, Xvs)
    rows.append((n, (pc == yva).float().mean().item(), (ps[yva != 4] == 4).float().mean().item()))
sc = pd.DataFrame(rows, columns=['model', 'acc ảnh sạch', 'ảnh không phải nhà thờ, đóng ô đỏ → đoán "nhà thờ"'])
table(sc, f'{PER_CLASS} ảnh train mỗi lớp, {len(va_ids)} ảnh validation ({time.time() - t:.0f} giây)', fmt={c: '{:.1%}' for c in sc.columns[1:]})
check('đường tắt kéo nhiều ảnh sang lớp nhà thờ hơn hẳn', sc.iloc[1, 2] > sc.iloc[0, 2] + 0.3)
note(f"Trên ảnh sạch, hai model gần như ngang nhau ({sc.iloc[0, 1]:.1%} và {sc.iloc[1, 1]:.1%}): <b>accuracy thông thường không lộ ra lỗi</b>. "
     f"Đóng ô đỏ lên ảnh không phải nhà thờ thì model có đường tắt đoán 'nhà thờ' <b>{sc.iloc[1, 2]:.0%}</b> số ảnh, model sạch chỉ {sc.iloc[0, 2]:.0%}.")

In [ ]:
def gradcam18(m, x, k):
    store = {}
    h = m.layer4.register_forward_hook(lambda mod, i, o: store.__setitem__('a', o))
    out = m(x[None].to(device)); h.remove()
    g = torch.autograd.grad(out[0, k], store['a'])[0]
    cam = F.relu((g.mean((2, 3), keepdim=True) * store['a']).sum(1))[0]
    cam = F.interpolate(cam[None, None], size=160, mode='bilinear', align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy(), out.softmax(1)[0, k].item()

ids = [int(np.flatnonzero(yva.numpy() == k)[0]) for k in [0, 6, 8]]
fig, axes = plt.subplots(2, 3, figsize=(12, 8.6))
for j, i in enumerate(ids):
    x = Xvs[i]
    for r, (n, m) in enumerate(models.items()):
        with torch.no_grad(): k = int(m(x[None].to(device)).argmax())
        cam, p = gradcam18(m, x, k)
        axes[r, j].imshow(show18(x)); axes[r, j].imshow(cam, cmap='magma', alpha=.55); axes[r, j].axis('off')
        axes[r, j].set_title(f'{n}\nđoán: {VI[k]} ({p:.2f}) · thật: {VI[yva[i]]}', fontsize=10, color=C['green'] if k == yva[i] else C['clay'])
fig.suptitle('Grad-CAM của lớp được đoán, trên ảnh mờ bị đóng ô đỏ', y=0.95)
plt.show()
note('Model có đường tắt đoán "nhà thờ" cho cả cá, xe, bóng, và Grad-CAM của nó dồn về góc có ô đỏ; model sạch không bị ô đỏ kéo đi. Grad-CAM lộ ra điều mà bảng accuracy trên ảnh sạch không cho thấy.')

## 5. Tóm tắt

In [ ]:
summary = [('Grad-CAM tương quan với occlusion (trung bình 3 ảnh)', f'{np.mean(corr):.2f}'),
           ('số lần chạy model: occlusion / Grad-CAM', f'{occ[0].size} / 1 forward + 1 backward'),
           ('acc ảnh sạch: model sạch / model có đường tắt', f'{sc.iloc[0, 1]:.1%} / {sc.iloc[1, 1]:.1%}'),
           ('ảnh đóng ô đỏ đoán thành nhà thờ: model sạch / model có đường tắt', f'{sc.iloc[0, 2]:.0%} / {sc.iloc[1, 2]:.0%}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 4 bỏ bước làm mờ (`T.Resize(20)`) rồi xem đường tắt còn mạnh không; đóng ô đỏ cho chỉ 50% ảnh nhà thờ; ở mục 1 đổi ô che 64 thành 32.